### Setup

In [ ]:
# Colab already ships a MATCHED torch + torchvision + sympy. Re-installing
# torchvision here can resolve a newer torchvision -> pull a different torch ->
# mismatched sympy, which breaks `import torch` with the lazy-import error
#   AttributeError: module 'sympy' has no attribute 'core'
# pyyaml / pillow are preinstalled too, so we install NOTHING that touches the
# torch stack. (If you already ran a version that reinstalled torchvision this
# session: Runtime -> "Disconnect and delete runtime" to reset, then run this.)
!pip install -q pyyaml

import os
import zipfile
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# --- CONFIGURATION ---
ZIP_PATH = "/content/drive/MyDrive/split_centroid_dataset/split_centroid_dataset.zip"
LOCAL_ROOT = "/content/dataset"
# ---------------------

# Extract Dataset
if os.path.exists(ZIP_PATH):
    print("Extracting Centroid Dataset...")
    if not os.path.exists(LOCAL_ROOT):
        os.makedirs(LOCAL_ROOT)
    !unzip -qo {ZIP_PATH} -d {LOCAL_ROOT}
    print("Extraction complete.")
else:
    print(f"Error: {ZIP_PATH} not found.")

### Architecture and Preprocessing

**v5 = the STN-FOMO backbone-sharpening effect, distilled into a fixed blur.**

We established three things empirically about STN-FOMO:

1. Its Spatial Transformer **does not rotate** — the localization net collapsed to a
   near-constant output (angle ~0 deg, std 0.07 deg over the test set), so it is not
   correcting scene rotation.
2. What it *does* apply is a **constant ~0.967x bilinear resample** via `grid_sample`.
   Replacing that resample with a true identity grid collapsed detection (crossing
   confidence dropped by ~0.17), so the resample is functionally load-bearing — the
   head is calibrated to the *blurred* features it produces.
3. Most importantly: feeding the **same 30 test images** through STN-FOMO's backbone
   (`add_7`) vs the architecturally-identical v4 backbone (`add_900`) gave wildly
   different feature statistics — STN-FOMO's backbone runs ~40x higher mean activation,
   ~7x larger std/max, and ~9x more spatial detail (total variation), at the same
   sparsity. **The STN-trained backbone is sharper and louder.**

The proposed mechanism: `grid_sample`'s bilinear interpolation is a mild **blur** that
averages neighbouring grid cells before the head sees them. A backbone trained
end-to-end *through* that blur is under constant pressure to **pre-compensate** —
encode features with larger dynamic range and finer spatial structure so enough
survives the blur. A backbone with a direct pass-through (v3/v4) feels no such
pressure and settles into a flatter, "good enough" representation. In other words the
blur is a built-in **adversarial bottleneck** during training.

**This notebook tests that mechanism in isolation.** It is the v3 network with the STN
*and its loc_fc localization branch removed entirely*, and a single **fixed,
non-learnable `grid_sample` blur** (constant scale ~0.967, zero rotation) inserted
between backbone and head. There is no rotation supervision, no auxiliary loss, no
learnable transform — just the blur. If the hypothesis holds, this alone should
reproduce STN-FOMO's sharper backbone and its accuracy margin over v3/v4.

The blur is present in **both training and inference** (the forward graph is identical
in eval), exactly like the STN in STN-FOMO, so it exports straight to ONNX as one
`GridSample` op. The dataset loader still accepts either centroid labels
(`cls x_c y_c`) or YOLO-OBB labels (`cls x1 y1 ...`), reducing OBB boxes to their
centroid, but no longer extracts rotation (the blur needs no targets).

In [ ]:
import math
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import MobileNet_V2_Weights
from PIL import Image
import glob
import yaml

# --- BACKBONE: full-width MobileNetV2 (alpha=1.0), ImageNet-pretrained ---
# Identical to v3 / STN-FOMO: full-width backbone (96 channels at block 13),
# width_mult=1.0. torchvision ships the ImageNet-pretrained weights.
def load_mobilenet_v2_100():
    """Return MobileNetV2 alpha=1.0 `.features`, ImageNet-pretrained (DEFAULT = V2)."""
    net = models.mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
    return net.features


# --- THE EXPERIMENT: a fixed, non-learnable blur bottleneck ---
# Constant ~0.967x uniform scale (the value STN-FOMO's collapsed STN converged to),
# zero rotation. We DROP the localization net entirely — there is nothing to learn
# here, only a fixed bilinear resample that blurs the 30x30 feature grid. The grid
# is precomputed once (H, W are fixed at 30) and broadcast over the batch, so this
# exports to ONNX as a single GridSample with a constant grid.
BLUR_SCALE = 0.967

class FixedGridSampleBlur(nn.Module):
    """Fixed, non-learnable bilinear resampling bottleneck.

    Replicates the measured EFFECT of STN-FOMO's GridSample without any
    localization net: a constant affine transform (uniform scale BLUR_SCALE,
    zero rotation) applied by bilinear grid_sample on the 30x30 feature grid.
    Because grid_sample interpolates between neighbouring cells, it acts as a
    mild blur the backbone must learn to pre-compensate for — the adversarial
    bottleneck hypothesised to push the backbone into a sharper, higher-
    dynamic-range regime. Present in BOTH training and inference, exactly like
    the STN in STN-FOMO (the head is calibrated to the blurred features).
    """
    def __init__(self, scale=BLUR_SCALE, grid_size=30):
        super().__init__()
        # theta = uniform scale, no rotation, no translation.
        theta = torch.tensor([[[scale, 0.0, 0.0],
                               [0.0, scale, 0.0]]], dtype=torch.float32)
        # Precompute the sampling grid for one example; broadcast over batch later.
        # align_corners=False matches torch's default STN export behaviour.
        grid = F.affine_grid(theta, (1, 1, grid_size, grid_size),
                             align_corners=False)  # (1, gs, gs, 2)
        self.register_buffer("base_grid", grid)

    def forward(self, x):
        b = x.shape[0]
        grid = self.base_grid.expand(b, -1, -1, -1)
        return F.grid_sample(x, grid, mode="bilinear",
                             padding_mode="border", align_corners=False)


# --- LOSS: RetinaNet focal with per-class alpha (identical to v3 / STN-FOMO) ---
# alpha controls the positive/negative balance per class:
#   high alpha -> model pushed to FIRE (fewer false negatives, more false positives)
#   low alpha  -> model pushed to SUPPRESS (fewer false positives, more false negatives)
FOCAL_GAMMA = 2.0
CLASS_ALPHA = {
    "railroad-crossing": 0.75,  # sparse, rotating, hard to hit; prioritise recall
    "lights-on":         0.50,  # visually similar to lights-off; 0.25 caused FNs
    "lights-off":        0.25,  # already ~100% recall at 0.25
    "trefolo":           0.25,  # already ~100% recall at 0.25
}


def per_cell_focal_loss(logits, target, alpha, gamma=FOCAL_GAMMA):
    """RetinaNet focal loss treating every grid cell as an independent binary
    classifier. logits, target: (B, C, H, W). alpha: (C,) per-class weight tensor.
    Normalized by the number of positive cells (RetinaNet-style)."""
    prob = torch.sigmoid(logits)
    ce   = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
    p_t     = prob * target + (1.0 - prob) * (1.0 - target)
    alpha_t = alpha.view(1, -1, 1, 1) * target + (1.0 - alpha.view(1, -1, 1, 1)) * (1.0 - target)
    loss    = alpha_t * (1.0 - p_t).pow(gamma) * ce
    num_pos = target.eq(1.0).sum().clamp(min=1.0)
    return loss.sum() / num_pos


def build_class_alpha(class_names):
    """Per-class focal alpha from CLASS_ALPHA table. Unknown classes default to 0.25."""
    alpha = [CLASS_ALPHA.get(n, 0.25) for n in class_names]
    return torch.tensor(alpha, dtype=torch.float32)


class FOMO_V5_480(nn.Module):
    """v3 architecture + a FIXED blur bottleneck, with NO localization branch.

    Full-width MobileNetV2 (alpha=1.0) cut at block 13 -> 30x30, 96 ch, then a
    fixed bilinear blur (FixedGridSampleBlur), then STN-FOMO's head: depthwise
    3x3 -> ReLU -> 1x1 (96->96) -> ReLU -> 1x1 (96->num_classes). One heatmap
    per class (no background); returns raw logits. The forward graph is identical
    in train and eval — the blur is always on."""
    def __init__(self, num_classes, blur_scale=BLUR_SCALE):
        super(FOMO_V5_480, self).__init__()
        backbone = load_mobilenet_v2_100()
        self.features = nn.Sequential(*list(backbone.children())[:14])
        backbone_channels = self._infer_backbone_channels()

        # The fixed adversarial blur bottleneck (replaces the learnable STN).
        self.blur = FixedGridSampleBlur(scale=blur_scale, grid_size=30)

        self.head = nn.Sequential(
            nn.Conv2d(backbone_channels, backbone_channels, kernel_size=3,
                      padding=1, groups=backbone_channels),   # depthwise 3x3
            nn.ReLU(),
            nn.Conv2d(backbone_channels, backbone_channels, kernel_size=1),  # pointwise
            nn.ReLU(),
            nn.Conv2d(backbone_channels, num_classes, kernel_size=1),        # classifier
        )

    def _infer_backbone_channels(self):
        was_training = self.training
        self.eval()
        with torch.no_grad():
            channels = self.features(torch.zeros(1, 3, 64, 64)).shape[1]
        self.train(was_training)
        return channels

    def forward(self, x):
        feat = self.features(x)
        feat = self.blur(feat)      # <-- fixed blur bottleneck, always on
        return self.head(feat)


class YOLOCentroidDataset(Dataset):
    """grid_size defaults to 30 (stride 16). Hard single-cell targets, no smearing.
    Accepts centroid OR YOLO-OBB labels (OBB -> centroid of the 4 corners). Unlike
    v3, no rotation target is produced — the fixed blur needs no supervision."""
    def __init__(self, split_root, num_classes, img_size=480, grid_size=30):
        self.img_dir = os.path.join(split_root, "images")
        self.label_dir = os.path.join(split_root, "labels")
        self.img_paths = sorted([p for p in glob.glob(os.path.join(self.img_dir, "*"))
                                  if p.lower().endswith(('.jpg', '.jpeg', '.png'))])
        self.num_classes = num_classes
        self.grid_size = grid_size
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        img = Image.open(img_path).convert('RGB')
        label_path = os.path.join(self.label_dir, os.path.splitext(os.path.basename(img_path))[0] + ".txt")

        gs = self.grid_size
        target = torch.zeros((self.num_classes, gs, gs), dtype=torch.float32)
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.split()
                    if len(parts) < 3: continue
                    cls_id = int(parts[0])
                    if not (0 <= cls_id < self.num_classes): continue
                    coords = [float(v) for v in parts[1:]]
                    if len(coords) >= 8:
                        # YOLO-OBB -> centroid = mean of the 4 corners.
                        xs, ys = coords[0:8:2], coords[1:8:2]
                        x_c, y_c = sum(xs) / 4.0, sum(ys) / 4.0
                    else:
                        x_c, y_c = coords[0], coords[1]   # centroid format
                    x_c = min(max(x_c, 0.0), 0.999999)
                    y_c = min(max(y_c, 0.0), 0.999999)
                    gx = min(int(x_c * gs), gs - 1)
                    gy = min(int(y_c * gs), gs - 1)
                    target[cls_id, gy, gx] = 1.0   # hard single-cell target
        return self.transform(img), target

### Finetuning

In [ ]:
# Folder name of the unzipped dataset under LOCAL_ROOT. Point this at your
# (split) OBB or centroid dataset, e.g. "obb_dataset" or "split_centroid_dataset".
DATASET_DIRNAME = "split_centroid_dataset"

def _resolve_split(dataset_root):
    """Return the folder holding images/ + labels/ for training (train/ split
    or a flat images/+labels/ layout)."""
    train_split = os.path.join(dataset_root, "train")
    if os.path.isdir(os.path.join(train_split, "images")):
        return train_split
    if os.path.isdir(os.path.join(dataset_root, "images")):
        return dataset_root
    raise FileNotFoundError(
        f"Could not find images/ under {dataset_root}/train or {dataset_root}."
    )

def run_training():
    dataset_root = os.path.join(LOCAL_ROOT, DATASET_DIRNAME)
    yaml_path = os.path.join(dataset_root, "data.yaml")
    with open(yaml_path, 'r') as f:
        data_cfg = yaml.safe_load(f)

    train_path = _resolve_split(dataset_root)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on: {device}")

    num_classes = len(data_cfg['names'])

    train_ds = YOLOCentroidDataset(train_path, num_classes, grid_size=30)
    print(f"Training images: {len(train_ds)} from {train_path}")
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)

    model = FOMO_V5_480(num_classes=num_classes).to(device)
    print(f"Blur bottleneck: fixed grid_sample, scale={BLUR_SCALE}, no localization net")

    class_alpha = build_class_alpha(data_cfg['names']).to(device)
    print(f"Focal alpha: {dict(zip(data_cfg['names'], class_alpha.tolist()))}")
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    epochs = 60
    for epoch in range(epochs):
        model.train()
        l_sum = 0
        for imgs, targs in train_loader:
            imgs, targs = imgs.to(device), targs.to(device)
            optimizer.zero_grad()
            preds = model(imgs)
            if preds.shape[-2:] != targs.shape[-2:]:
                raise RuntimeError(
                    f"Grid mismatch: model output {tuple(preds.shape[-2:])} vs target "
                    f"{tuple(targs.shape[-2:])}. Set grid_size to the model's output resolution."
                )
            loss = per_cell_focal_loss(preds, targs, class_alpha)
            loss.backward()
            optimizer.step()
            l_sum += loss.item()

        print(f"Epoch {epoch+1}/{epochs} - Det Loss: {l_sum/len(train_loader):.4f}")

    save_dir = "/content/drive/MyDrive/fomo_results"
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, "fomo_v5_480.pt")
    torch.save(model.state_dict(), save_path)
    print(f"Model saved to: {save_path}")

run_training()